In [2]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List, Optional
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool
from langgraph.types import Command, interrupt

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

documents = [
    "LangChain is a framework for building LLM-powered applications using chains and prompts.",
    "LangGraph extends LangChain with stateful graph-based workflows and cycles.",
    "LangChain supports tools, memory, and retrieval out of the box.",
    "LangGraph uses StateGraph with nodes and edges for complex agent architectures.",
    "LangChain was created by Harrison Chase in late 2022.",
    "LangGraph was released in early 2024 as part of the LangChain ecosystem.",
    "RAG combines document retrieval with LLM generation for knowledge-grounded answers.",
    "Vector databases like FAISS and Pinecone store embeddings for semantic search.",
    "Transformers use attention mechanisms and are the backbone of modern LLMs.",
    "Fine-tuning adapts a pre-trained model to a specific domain using labeled data.",
]
system_prompt = """You are a decompose of question specialist. Look at the user question and if it belongs to multiple context then split to multiple question and return as list of questions.
  Even if it's single category or context then return single question as list
"""
answer_system_prompt="You are answer evaluator of question.You checked the answer provided by the process is relevant or not. If relevent set the value of is_relevant to True"

compr_answer_system_prompt="You are evaluate all answers of all sub questions and generate the final answers"
class MultiHopState(TypedDict):
    question:       str                                    # original complex question
    sub_questions:  List[str]                              # decomposed sub-questions
    sub_answers:    Annotated[List[str], operator.add]     # accumulates answers via reducer
    current_index:  int                                    # which sub-question we're working on
    documents:      List[str]                              # retrieved docs for current sub-question
    final_answer:   str

class SubQuestions(BaseModel):
    questions: List[str]

class SubAnswers(BaseModel):
    is_relevant: bool

def decompose_node(state:MultiHopState):
    msg=state["question"]
    struc_resp=llm_openai.with_structured_output(SubQuestions)
    response = struc_resp.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"User request: {msg}")
    ])
    return {"sub_questions":response.questions,"current_index":0}

def retrieve_node(state:MultiHopState):
    sub_q = state["sub_questions"][state["current_index"]].lower()
    relevant = [d for d in documents if any(w in d.lower() for w in sub_q.split())]
    return {"documents": relevant if relevant else documents[:2]}

def answer_sub_node(state: MultiHopState):
    sub_q = state["sub_questions"][state["current_index"]]
    docs_text = "\n".join(state["documents"])
    response = llm_openai.invoke([
        SystemMessage(content=f"Answer the question using only these documents:\n\n{docs_text}"),
        HumanMessage(content=sub_q)
    ])
    return {
        "sub_answers": [f"Q: {sub_q}\nA: {response.content}"],   # one item → reducer appends it
        "current_index": state["current_index"] + 1
    }

def route_or_continue(state: MultiHopState) -> str:
    current_index = state["current_index"]
    sub_questions=state["sub_questions"]
    if current_index < len(sub_questions): return "retrieve"
    else:
        return "synthesize"

def synthesize_node(state:MultiHopState):
    questions = state["sub_questions"]
    answers = state["sub_answers"]
    answers_text = "\n\n".join(state["sub_answers"])
    response = llm_openai.invoke([
                SystemMessage(content=compr_answer_system_prompt),
                HumanMessage(content=f"Original question: {state['question']}\n\nResearch findings:\n{answers_text}")
            ])

    return {"final_answer": response.content}



graph=StateGraph(MultiHopState)
graph.add_node("decompose_node", decompose_node)
graph.add_node("retrieve_node", retrieve_node)
graph.add_node("answer_sub_node", answer_sub_node)
graph.add_node("synthesize_node", synthesize_node)
graph.add_edge(START, "decompose_node")
graph.add_edge("decompose_node", "retrieve_node")
graph.add_edge("retrieve_node", "answer_sub_node")
graph.add_conditional_edges("answer_sub_node", route_or_continue,
    {"retrieve": "retrieve_node", "synthesize": "synthesize_node"})
graph.add_edge("synthesize_node", END)
app = graph.compile()

r1 = app.invoke({
    "question": "Compare LangChain and LangGraph - when were they created and how do they differ?",
    "sub_questions": [], "sub_answers": [], "current_index": 0,
    "documents": [], "final_answer": ""
})
r2 = app.invoke({
    "question": "What are the main approaches to giving LLMs access to external knowledge?",
    "sub_questions": [], "sub_answers": [], "current_index": 0,
    "documents": [], "final_answer": ""
})

for r, q in [(r1, "Compare"), (r2, "Approaches")]:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    print(f"Sub-questions: {r['sub_questions']}")
    print(f"Sub-answers count: {len(r['sub_answers'])}")
    print(f"Final: {r['final_answer'][:200]}")






C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SubQuestions(questions=['...and LangGraph differ?']), input_type=SubQuestions])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SubQuestions(questions=['...o external knowledge?']), input_type=SubQuestions])
  return self.__pydantic_serializer__.to_python(



Q: Compare
Sub-questions: ['When were LangChain and LangGraph created?', 'How do LangChain and LangGraph differ?']
Sub-answers count: 2
Final: **LangChain vs. LangGraph – creation dates & core differences**

| Aspect | LangChain | LangGraph |
|--------|-----------|-----------|
| **First released** | **Late 2022** – created by Harrison Chase 

Q: Approaches
Sub-questions: ['What are the main approaches to giving LLMs access to external knowledge?']
Sub-answers count: 1
Final: **Main approaches for giving large language models (LLMs) access to external knowledge**

| Approach | How it works | Typical use‑cases |
|----------|--------------|-------------------|
| **1. Retriev
